# Notebook 12 — Workshop Cleanup

Removes all assets created by notebooks 01–11 so you can start fresh or leave the workspace clean.

**What gets deleted:**

| Step | Assets | Created by |
|------|--------|------------|
| 1 | Genie spaces (5) | Notebooks 03, 08, 11 |
| 2 | Databricks App | Notebook 09 |
| 3 | Workspace files (app code, skill file) | Notebooks 06, 09 |
| 4 | Tables (9), view (1), functions (3) | Notebooks 02, 03, 07, 08 |
| 5 | Volume + prebuild data | Notebook 01 |

**What is NOT deleted:** Your catalog and schema remain intact (empty).

**Before you start:** Set `confirm_cleanup` to `yes` in the widget at the top of this notebook. Without confirmation, every destructive cell is skipped.

## Load config

In [ ]:
%run ./00_workshop_config

## Confirmation guard

Set the widget to **yes** to enable cleanup. Default is **no** (safe).

In [ ]:
dbutils.widgets.dropdown("confirm_cleanup", "no", ["no", "yes"], "Confirm cleanup")
CONFIRMED = dbutils.widgets.get("confirm_cleanup") == "yes"

if CONFIRMED:
    print(f"Cleanup ENABLED for {CATALOG}.{SCHEMA}")
else:
    print("Cleanup DISABLED. Set the confirm_cleanup widget to 'yes' and re-run.")

## Step 1 — Delete Genie spaces

Removes the 5 workshop Genie spaces by title match. Spaces not matching these titles are left untouched.

In [ ]:
import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}

WORKSHOP_SPACE_TITLES = {
    f"{GENIE_SPACE_PREFIX} - Configured",
    f"{GENIE_SPACE_PREFIX} - No Examples",
    f"{GENIE_SPACE_PREFIX} - Blank",
    f"{GENIE_SPACE_PREFIX} - Security Demo",
    f"{GENIE_SPACE_PREFIX} - Genie Code Built",
    f"[CICD] {GENIE_SPACE_PREFIX} - Configured",
}

if not CONFIRMED:
    print("Skipped (not confirmed).")
else:
    resp = requests.get(f"{host}/api/2.0/genie/spaces", headers=headers)
    deleted = 0
    if resp.status_code == 200:
        for s in resp.json().get("spaces", []):
            title = s.get("title") or s.get("display_name", "")
            if title in WORKSHOP_SPACE_TITLES:
                sid = s.get("space_id") or s.get("id")
                dr = requests.delete(f"{host}/api/2.0/genie/spaces/{sid}", headers=headers)
                status = "deleted" if dr.status_code in (200, 204) else f"failed ({dr.status_code})"
                print(f"  {title}: {status}")
                deleted += 1
    print(f"\nDeleted {deleted} Genie space(s).")

## Step 2 — Delete Databricks App

Removes the `manufacturing-genie-app` created by notebook 09.

In [ ]:
if not CONFIRMED:
    print("Skipped (not confirmed).")
else:
    for app_name in ["manufacturing-genie-app", "manufacturing-genie"]:
        dr = requests.delete(f"{host}/api/2.0/apps/{app_name}", headers=headers)
        if dr.status_code in (200, 204):
            print(f"  Deleted app: {app_name}")
        elif dr.status_code == 404:
            print(f"  App not found: {app_name} (already gone)")
        else:
            print(f"  Delete {app_name}: {dr.status_code} {dr.text[:100]}")

## Step 3 — Delete workspace files

Removes the app code folder and the Genie skill file uploaded by notebooks 06 and 09.

In [ ]:
if not CONFIRMED:
    print("Skipped (not confirmed).")
else:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    user_email = ctx.userName().get()

    paths_to_delete = [
        f"/Workspace/Users/{user_email}/manufacturing-genie-app",
        f"/Workspace/Users/{user_email}/.assistant/skills/manufacturing-genie-context.md",
    ]

    for path in paths_to_delete:
        dr = requests.post(
            f"{host}/api/2.0/workspace/delete",
            headers=headers,
            json={"path": path, "recursive": True},
        )
        if dr.status_code in (200, 204):
            print(f"  Deleted: {path}")
        elif dr.status_code == 404:
            print(f"  Not found: {path} (already gone)")
        else:
            print(f"  {path}: {dr.status_code} {dr.text[:100]}")

## Step 4 — Drop Unity Catalog objects

Drops the view, 9 tables, and 3 SQL functions created by the workshop. Uses `IF EXISTS` so it is safe to re-run.

In [ ]:
if not CONFIRMED:
    print("Skipped (not confirmed).")
else:
    drops = [
        f"DROP VIEW IF EXISTS {fqn}.production_events_restricted",
        f"DROP TABLE IF EXISTS {fqn}.plants",
        f"DROP TABLE IF EXISTS {fqn}.production_lines",
        f"DROP TABLE IF EXISTS {fqn}.operators",
        f"DROP TABLE IF EXISTS {fqn}.production_events",
        f"DROP TABLE IF EXISTS {fqn}.quality_metrics_daily",
        f"DROP TABLE IF EXISTS {fqn}.safety_incidents",
        f"DROP TABLE IF EXISTS {fqn}.equipment_feedback",
        f"DROP TABLE IF EXISTS {fqn}.workshop_config",
        f"DROP TABLE IF EXISTS {fqn}.genie_benchmark_runs",
        f"DROP FUNCTION IF EXISTS {fqn}.compute_defect_rate",
        f"DROP FUNCTION IF EXISTS {fqn}.compute_oee_summary",
        f"DROP FUNCTION IF EXISTS {fqn}.get_best_production_line",
    ]

    for stmt in drops:
        try:
            spark.sql(stmt)
            obj = stmt.split("IF EXISTS ")[1]
            print(f"  Dropped: {obj}")
        except Exception as e:
            print(f"  {stmt}: {str(e)[:80]}")

## Step 5 — Drop Volume and prebuild data

Removes the `workshop_data` volume and all Parquet files inside it.

In [ ]:
if not CONFIRMED:
    print("Skipped (not confirmed).")
else:
    try:
        spark.sql(f"DROP VOLUME IF EXISTS {CATALOG}.{SCHEMA}.{PREBUILD_VOLUME}")
        print(f"  Dropped volume: {CATALOG}.{SCHEMA}.{PREBUILD_VOLUME}")
    except Exception as e:
        print(f"  Volume drop: {str(e)[:80]}")

## Summary

All workshop assets have been removed. Your catalog (`{CATALOG}`) and schema (`{SCHEMA}`) still exist but are now empty.

To re-run the workshop, start from notebook **01**.

In [ ]:
if CONFIRMED:
    print("="*50)
    print("Workshop cleanup complete.")
    print("="*50)
    print(f"Catalog : {CATALOG} (kept)")
    print(f"Schema  : {SCHEMA} (kept, now empty)")
    print(f"\nTo re-run the workshop, start from notebook 01.")
else:
    print("No changes made. Set confirm_cleanup = yes to run cleanup.")